In [1]:

import os, time, numpy as np
os.chdir('/workspace/cf2d9721-21a4-45f3-a38f-f2135c4067b0')
# Check for the Odlyzko file
for root, dirs, files in os.walk('.'):
 for f in files:
 if 'zeta' in f.lower() or 'odlyzko' in f.lower() or 'zeros' in f.lower():
 print(os.path.join(root, f), os.path.getsize(os.path.join(root, f)))


./zeta_zeros_N100000_dps9.npy 800128


In [2]:

zeros_low = np.load('zeta_zeros_N100000_dps9.npy')
print(zeros_low.shape, zeros_low.dtype)
print(zeros_low[:3])
# R2 gate validation
spec = [14.134725141, 21.022039639, 25.010857580]
diffs = [abs(zeros_low[i] - spec[i]) for i in range(3)]
print("max abs diff vs spec:", max(diffs))


(100000,) float64
[14.13472514 21.02203964 25.01085758]
max abs diff vs spec: 1.000000082740371e-09


In [3]:

import mpmath as mp
mp.mp.dps = 50

# Define Hardy Z for sign-change-free refinement, but easier: use mp.zetazero with starting hint? 
# mp.zetazero(n) computes the n-th zero - but that's de novo and expensive.
# We want REFINEMENT from a seed. Use findroot on zeta(0.5 + i*t) for t real => use the real part or imaginary part?
# Actually we want roots of zeta(1/2 + i*t)=0. Since zeta is complex, use Hardy Z(t) which is real-valued on critical line.
# Z(t) = exp(i*theta(t)) * zeta(1/2 + i*t) is real. Use mp.siegelz.

# Test refinement on first zero
t0 = mp.mpf(str(zeros_low[0]))
print("seed:", t0)
t_ref = mp.findroot(mp.siegelz, t0)
print("refined:", t_ref)


seed: 14.134725142
refined: 14.134725141734693790457251983562470270784257115699


In [4]:

# Good. Now refine all 5000 zeros. Time a small batch first.
N = 5000
seeds = zeros_low[:N]

t_start = time.time()
batch = []
for i in range(20):
 t = mp.findroot(mp.siegelz, mp.mpf(str(seeds[i])))
 batch.append(t)
elapsed = time.time() - t_start
print(f"20 zeros refined in {elapsed:.2f}s => ~{elapsed/20*1000:.1f} ms/zero, est total {elapsed/20*N/60:.1f} min")


20 zeros refined in 0.36s => ~17.9 ms/zero, est total 1.5 min


In [5]:

# Excellent - very fast. Let's refine all 5000.
t_start = time.time()
refined = []
for i, s in enumerate(seeds):
 t = mp.findroot(mp.siegelz, mp.mpf(str(s)))
 refined.append(t)
 if (i+1) % 1000 == 0:
 print(f" {i+1}/{N} in {time.time()-t_start:.1f}s")
elapsed = time.time() - t_start
print(f"Total: {elapsed:.1f}s for {N} zeros")


TimeoutError: Code execution timed out after 900.0 seconds

In [6]:

# Timed out at 900s. The per-zero cost grows with t (siegelz cost grows with t).
# Let me check progress - the variable `refined` should still hold partial.
print(len(refined), "zeros refined before timeout")
print("last index:", len(refined)-1, "value:", refined[-1])


2176 zeros refined before timeout
last index: 2175 value: 2698.2909613213957824951012700274909100777847423507


In [7]:

# We have 2176 / 5000. Need to continue from index 2176 onward.
# Save progress so far in case of another issue.
np.save('zeta_refined_partial.npy', np.array([str(r) for r in refined]))

# Continue from 2176. Time progress with periodic saves.
t_start = time.time()
start_idx = len(refined)
chunk_size = 500
print(f"Continuing from index {start_idx}")
for i in range(start_idx, min(start_idx + chunk_size, N)):
 t = mp.findroot(mp.siegelz, mp.mpf(str(seeds[i])))
 refined.append(t)
elapsed = time.time() - t_start
print(f"Refined {chunk_size} more in {elapsed:.1f}s (total now {len(refined)})")
np.save('zeta_refined_partial.npy', np.array([str(r) for r in refined]))


Continuing from index 2176


Refined 500 more in 353.7s (total now 2676)


In [8]:

# ~0.7 s/zero now. Remaining 5000-2676 = 2324 will take ~1600s. We're at ~1500s total runtime; budget is 3600s.
# Need to be more efficient. The bottleneck is mp.siegelz at high t with dps=50.
# Approach: refine in chunks with periodic saves so we don't lose progress.
t_start = time.time()
chunk_size = 800
start_idx = len(refined)
for i in range(start_idx, min(start_idx + chunk_size, N)):
 t = mp.findroot(mp.siegelz, mp.mpf(str(seeds[i])))
 refined.append(t)
elapsed = time.time() - t_start
print(f"Refined {chunk_size} more in {elapsed:.1f}s (total now {len(refined)})")
np.save('zeta_refined_partial.npy', np.array([str(r) for r in refined]))


Refined 800 more in 577.5s (total now 3476)


In [9]:

# 1524 zeros remaining at ~0.72s/zero => ~1100s. Continue.
t_start = time.time()
start_idx = len(refined)
remaining = N - start_idx
for i in range(start_idx, N):
 t = mp.findroot(mp.siegelz, mp.mpf(str(seeds[i])))
 refined.append(t)
 if (i+1) % 500 == 0:
 np.save('zeta_refined_partial.npy', np.array([str(r) for r in refined]))
 print(f" {i+1}/{N} in {time.time()-t_start:.1f}s elapsed in this chunk")
elapsed = time.time() - t_start
print(f"Refined remaining {remaining} in {elapsed:.1f}s; total = {len(refined)}")
np.save('zeta_refined_partial.npy', np.array([str(r) for r in refined]))


TimeoutError: Code execution timed out after 900.0 seconds

In [10]:

# Check progress
print(len(refined), "zeros refined")
# Save what we have
np.save('zeta_refined_partial.npy', np.array([str(r) for r in refined]))
print("Last index:", len(refined)-1, "value:", refined[-1])


4705 zeros refined
Last index: 4704 value: 5172.8016888767915581944680115461974926308484781299


In [11]:

# Only 295 zeros remaining. At ~0.85 s/zero => ~250s. Let's continue but with a smaller chunk to avoid 900s timeout.
t_start = time.time()
start_idx = len(refined)
chunk = min(295, N - start_idx)
for i in range(start_idx, start_idx + chunk):
 t = mp.findroot(mp.siegelz, mp.mpf(str(seeds[i])))
 refined.append(t)
elapsed = time.time() - t_start
print(f"Refined {chunk} more in {elapsed:.1f}s; total = {len(refined)}")
np.save('zeta_refined_partial.npy', np.array([str(r) for r in refined]))


TimeoutError: Code execution timed out after 194.0 seconds